# 38 - Complete script dependency inventory

Read-only structural scan of every tracked Python script under scripts/, 10 September 2026.
No experiment is run. Dispositions are conservative triage from imports and utility-name
hints, not a full semantic review or deletion approval. Notebook mentions do not prove
saved-output coverage. Literal artifact paths do not prove completed execution; dynamic
paths may be unresolved. Historical source hashes are not rewritten.


In [1]:
from pathlib import Path
import ast, json, hashlib, re, collections, html, subprocess
from IPython.display import display, HTML
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists())
tracked=subprocess.check_output(['git','ls-files'],cwd=ROOT,text=True).splitlines()
py=[p for p in tracked if p.endswith('.py') and (ROOT/p).exists()]
scripts=[p for p in py if p.startswith('scripts/')]
modules={p[:-3].replace('/','.'):p for p in py}
stems=collections.defaultdict(list)
for p in py:stems[Path(p).stem].append(p)
inbound=collections.defaultdict(set);trees={};errors=[];dynamic=collections.defaultdict(list)
for p in py:
    try:trees[p]=ast.parse((ROOT/p).read_text(encoding='utf-8-sig'))
    except (SyntaxError,UnicodeError) as e:errors.append({'file':p,'error':str(e)});continue
    for node in ast.walk(trees[p]):
        names=[]
        if isinstance(node,ast.Import):names=[a.name for a in node.names]
        if isinstance(node,ast.ImportFrom):names=[node.module or '']+[(node.module+'.' if node.module else '')+a.name for a in node.names]
        for name in names:
            for target in ([modules[name]] if name in modules else stems.get(name,[])):
                if target!=p:inbound[target].add(p)
        if isinstance(node,ast.Call) and any(x in ast.unparse(node.func) for x in ['import_module','spec_from_file_location','run_path','subprocess','os.system']):dynamic[p].append(str(node.lineno)+': '+ast.unparse(node.func))
books={p:json.loads((ROOT/p).read_text(encoding='utf-8')) for p in tracked if p.startswith('notebooks/') and p.endswith('.ipynb') and (ROOT/p).exists()}
rows=[]
for p in sorted(scripts):
    tree=trees.get(p);text=(ROOT/p).read_text(encoding='utf-8-sig')
    purpose=(ast.get_docstring(tree) or 'No module docstring').splitlines()[0] if tree else 'Parse error'
    mentions=[name for name,b in books.items() if any(p in (c['source'] if isinstance(c['source'],str) else ''.join(c['source'])) for c in b['cells'])]
    literals=sorted(set(v for node in ast.walk(tree) if isinstance(node,ast.Constant) and isinstance(node.value,str) for v in re.findall(r'data/(?:processed|interim|archive|raw)/[A-Za-z0-9_./-]+',node.value))) if tree else []
    consumers=sorted(inbound[p]);utility=Path(p).stem.startswith(('download_','acquire_','package_','render_','build_','verify_','freeze_','finalize_'))
    disposition='Extract shared functions before migration' if consumers else ('Retain utility; inspect scope' if utility else 'Notebook candidate; manual evidence review required')
    rows.append(dict(script=p,purpose=purpose,disposition=disposition,consumers=consumers,notebook_mentions=mentions,existing_literal_artifacts=[v for v in literals if (ROOT/v).exists()],unresolved_literal_paths=[v for v in literals if not (ROOT/v).exists()],dynamic_calls=dynamic[p],sha256=hashlib.sha256((ROOT/p).read_bytes()).hexdigest()))
summary=dict(scripts=len(rows),python_files_parsed=len(trees),parse_errors=errors,notebook_files=len(books),dispositions=dict(collections.Counter(r['disposition'] for r in rows)),scripts_with_notebook_mentions=sum(bool(r['notebook_mentions']) for r in rows),scripts_with_existing_literal_artifacts=sum(bool(r['existing_literal_artifacts']) for r in rows),scripts_with_unresolved_literal_paths=sum(bool(r['unresolved_literal_paths']) for r in rows),scripts_with_dynamic_calls=sum(bool(r['dynamic_calls']) for r in rows))
print(json.dumps(summary,indent=2))
def show(records,fields):
    text='<div style="overflow:auto;max-height:650px"><table><tr>'+''.join('<th>'+html.escape(f)+'</th>' for f in fields)+'</tr>'
    for row in records:text+='<tr>'+''.join('<td>'+html.escape(str(row.get(f,'')))+'</td>' for f in fields)+'</tr>'
    display(HTML(text+'</table></div>'))


{
  "scripts": 139,
  "python_files_parsed": 216,
  "parse_errors": [],
  "notebook_files": 54,
  "dispositions": {
    "Notebook candidate; manual evidence review required": 92,
    "Extract shared functions before migration": 27,
    "Retain utility; inspect scope": 20
  },
  "scripts_with_notebook_mentions": 11,
  "scripts_with_existing_literal_artifacts": 71,
  "scripts_with_unresolved_literal_paths": 8,
  "scripts_with_dynamic_calls": 2
}


## Every script and its consumers

In [2]:
show(rows,['script','purpose','disposition','consumers','notebook_mentions'])

script,purpose,disposition,consumers,notebook_mentions
scripts/ablate_tier1_healthy_enrichment.py,No module docstring,Notebook candidate; manual evidence review required,[],[]
scripts/acquire_tvs_schema_samples.py,"Acquire two size-selected TVS laboratory schema samples, not whole cohorts.",Extract shared functions before migration,"['scripts/run_tvs_locked_pilot.py', 'scripts/screen_tvs_lab_metadata.py']",[]
scripts/analyze_lower_back_vs_three_channel_external_errors.py,No module docstring,Notebook candidate; manual evidence review required,[],[]
scripts/analyze_sint_revalexo_paired_errors.py,Paired participant-level comparison of original vs expanded Inception.,Notebook candidate; manual evidence review required,[],[]
scripts/audit_development_raw_axis_recovery.py,Inventory raw tri-axial development files before rebuilding the raw-axis contract.,Notebook candidate; manual evidence review required,[],[]
scripts/audit_gaitex_affine_adapter.py,"Test a leakage-safe, waveform-preserving GAITEX source adapter.",Notebook candidate; manual evidence review required,[],[]
scripts/audit_gaitex_synthesis_source.py,Audit GAITEX as a physics-grounded virtual-IMU synthesis source.,Notebook candidate; manual evidence review required,[],[]
scripts/audit_gaitex_virtual_training_contract.py,Compare GAITEX virtual signals with the real Felius/Voisard healthy contract.,Notebook candidate; manual evidence review required,[],[]
scripts/audit_kiel_validation_dataset.py,Audit the public healthy subset of the Kiel Validation Dataset.,Notebook candidate; manual evidence review required,[],[]
scripts/audit_marea_synthetic_quality.py,No module docstring,Notebook candidate; manual evidence review required,[],[]


## Artifact-path and dynamic-call flags

In [3]:
show([r for r in rows if r['unresolved_literal_paths'] or r['dynamic_calls']],['script','existing_literal_artifacts','unresolved_literal_paths','dynamic_calls'])

script,existing_literal_artifacts,unresolved_literal_paths,dynamic_calls
scripts/audit_triaxial_healthy_domain.py,['data/processed/triaxial_healthy_domain_audit.csv'],['data/raw/triaxial_accelerometer/extracted'],[]
scripts/audit_zenodo_stroke_rehab.py,['data/processed/zenodo_stroke_rehab_file_audit.csv'],['data/raw/zenodo_stroke_rehab/extracted'],[]
scripts/benchmark_mobilise_d_bout_cohorts.py,['data/raw/mobilise_d_cvs/walking_bout_dmo/extracted'],['data/raw/mobilise_d_cvs/main_datasets/Main'],[]
scripts/benchmark_mobilise_d_clinical_cohorts.py,[],['data/raw/mobilise_d_cvs/main_datasets/Main'],[]
scripts/classification/probe_eldernet_lower_back.py,"['data/processed/bilateral_phase_v1/trial_features.csv', 'data/processed/directional_hr_v1/participants_main.csv', 'data/processed/eldernet_lower_back_v1', 'data/raw/voisard_2025/data']",[],['41: importlib.util.spec_from_file_location']
scripts/extract_duogait_full_gait_cycles.py,"['data/processed/duogait_full_gait_cycle_metadata.csv', 'data/processed/duogait_full_gait_cycles_float32.npy']",['data/raw/duogait_2023/data'],[]
scripts/extract_marea_full_gait_cycles.py,"['data/processed/marea_full_gait_cycle_metadata.csv', 'data/processed/marea_full_gait_cycles_float32.npy']",['data/raw/marea_2017/data'],[]
scripts/materialize_tier1_healthy_windows.py,[],"['data/raw/duogait_2023/data/repository_interim/OG_st_control', 'data/raw/marea_2017/data']",[]
scripts/materialize_zenodo_stroke_windows.py,[],['data/raw/zenodo_stroke_rehab/extracted/interim'],[]
scripts/verify_lower_back_release.py,[],[],['47: subprocess.run']


## Source provenance and exact duplicates

In [4]:
hashes=collections.defaultdict(list)
for row in rows:hashes[row['sha256']].append(row['script'])
print('Byte-identical script groups:',[v for v in hashes.values() if len(v)>1])
show(rows,['script','sha256'])
OUT=ROOT/'data/processed/git_review_2026-09-09'
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'all_script_inventory.json').write_text(json.dumps(dict(summary=summary,scripts=rows),indent=2),encoding='utf-8')


Byte-identical script groups: []


script,sha256
scripts/ablate_tier1_healthy_enrichment.py,504db18b3de72e4fac6c7baac88f1d9f1ca681de9cd56004d623e4face9109eb
scripts/acquire_tvs_schema_samples.py,aa2f21402747a7bb56e96b2166cf34596058297ef446360d3688557759cc01c1
scripts/analyze_lower_back_vs_three_channel_external_errors.py,c013ab1a56224ddcec94acf2da0d9c1892130a49154b993417edaae436ae1157
scripts/analyze_sint_revalexo_paired_errors.py,4061ae304792c239594526b6b17bfc0891d4257d52a5593b352bec80ffbd36a1
scripts/audit_development_raw_axis_recovery.py,a87bd5986877021f471bce4afbb89755f9f0daaaa65af9983b9e520383388d20
scripts/audit_gaitex_affine_adapter.py,a453ba2eab4527ed4c83b15d097deaa2c967c45866489d5502bcdbfa5bcc95fd
scripts/audit_gaitex_synthesis_source.py,6954822174520f59ef9d897d5431bc4f70d86e26b50bf1e87bd5f7578b6286e5
scripts/audit_gaitex_virtual_training_contract.py,aadddc73afc8294d57bcd6fa3eeefa501ff46d008ae9963499f3ddef027aa7a7
scripts/audit_kiel_validation_dataset.py,1ec2704c0d1c29466d32cd376b8021c2bb3fa0d02ca10dee9e44353cf8c58ec5
scripts/audit_marea_synthetic_quality.py,bd3a7a5b0329256b242834faaf32f10b67121605bf75e8cbfac45f8518cb281f


81226

## Missing raw-path follow-up

Existing archive counterparts confirm relocation for six of the eight flagged scripts. Two Mobilise-D scripts have no matching path at either location checked. This is a path check, not an acquisition request or a reason to rerun old benchmarks.

In [5]:
show([{'script':r['script'],'old_path':v,'archive_counterpart':v.replace('data/raw/','data/archive/raw/',1),'archive_exists':(ROOT/v.replace('data/raw/','data/archive/raw/',1)).exists()} for r in rows for v in r['unresolved_literal_paths']],['script','old_path','archive_counterpart','archive_exists'])

script,old_path,archive_counterpart,archive_exists
scripts/audit_triaxial_healthy_domain.py,data/raw/triaxial_accelerometer/extracted,data/archive/raw/triaxial_accelerometer/extracted,True
scripts/audit_zenodo_stroke_rehab.py,data/raw/zenodo_stroke_rehab/extracted,data/archive/raw/zenodo_stroke_rehab/extracted,True
scripts/benchmark_mobilise_d_bout_cohorts.py,data/raw/mobilise_d_cvs/main_datasets/Main,data/archive/raw/mobilise_d_cvs/main_datasets/Main,False
scripts/benchmark_mobilise_d_clinical_cohorts.py,data/raw/mobilise_d_cvs/main_datasets/Main,data/archive/raw/mobilise_d_cvs/main_datasets/Main,False
scripts/extract_duogait_full_gait_cycles.py,data/raw/duogait_2023/data,data/archive/raw/duogait_2023/data,True
scripts/extract_marea_full_gait_cycles.py,data/raw/marea_2017/data,data/archive/raw/marea_2017/data,True
scripts/materialize_tier1_healthy_windows.py,data/raw/duogait_2023/data/repository_interim/OG_st_control,data/archive/raw/duogait_2023/data/repository_interim/OG_st_control,True
scripts/materialize_tier1_healthy_windows.py,data/raw/marea_2017/data,data/archive/raw/marea_2017/data,True
scripts/materialize_zenodo_stroke_windows.py,data/raw/zenodo_stroke_rehab/extracted/interim,data/archive/raw/zenodo_stroke_rehab/extracted/interim,True
